In [1]:
import sys
sys.path.append('../')

import contextlib
import io
import time

import pandas as pd

from utilities import read_instance, read_solution, is_feasible, calculate_cost, score, size_class
from clarke_wright_solution import run_cw_ls_algo, time_budget
from route_first_cluster_second_solution import run_rfcs_algo
from run_benchmarks import find_instances

In [2]:
# Each solver takes (demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit) -> (cost, routes)
SOLVERS = {
    'cw_ls': lambda demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit: run_cw_ls_algo(demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit=time_limit),
    'rfcs': lambda demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit: run_rfcs_algo(demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit=time_limit),
}

In [8]:
def run_benchmarks_df(solvers=SOLVERS, sets=('A', 'B', 'P', 'X'), samples_limit=None, time_vehicle_capacity=None):
    '''Runs every solver on the benchmark instances and returns one row per (solver, instance).'''
    rows = []
    for s in sets:

        # limit the number of samples if needed
        instances = list(find_instances([s]))
        if samples_limit:
            instances= instances[:samples_limit]

        for path, opt_path in instances:
            # read the test case and provided solution
            demand, dist_matrix, vehicle_capacity, num_vehicles = read_instance(path)
            optimal_cost = read_solution(opt_path)[0] if opt_path else None

            num_customers = len(demand) - 1
            for name, solver in solvers.items():
                time_start = time.perf_counter()
                _, routes = solver(demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit=None)
                elapsed = time.perf_counter() - time_start

                with contextlib.redirect_stdout(io.StringIO()):   # hide is_feasible's message
                    feasible = is_feasible(routes, demand, vehicle_capacity, num_vehicles)
                cost = calculate_cost(routes, dist_matrix)

                rows.append({
                    # instance variables
                    'solver': name, 'set': s, 'instance': path.stem,
                    'num_customers': num_customers, 'vehicle_capacity': vehicle_capacity, 'num_vehicles': num_vehicles,
                    'total_demandand': sum(demand),
                    # results
                    'time_taken': elapsed, 'cost': cost,
                    'optimal_cost': optimal_cost,
                    'gap_pct': 100 * (cost - optimal_cost) / optimal_cost if optimal_cost else None,
                    'feasible': feasible,
                })
                # print(f'{name:<6} {path.stem:<16} cost={cost:10.1f}  time={elapsed:6.1f}s', flush=True)
    return pd.DataFrame(rows)


# Quick run: first 3 instances of set A, vehicle_capacityped at 5 s each. Drop the limits for the full run.
results = run_benchmarks_df(sets=['A'], samples_limit=None, time_vehicle_capacity=5)
results

,solver,set,instance,num_customers,vehicle_capacity,num_vehicles,total_demandand,time_taken,cost,optimal_cost,gap_pct,feasible
0,cw_ls,A,A-n32-k5,31,100,5,410,0.012307,827,784,5.484694,True
1,rfcs,A,A-n32-k5,31,100,5,410,0.046904,798,784,1.785714,True
2,cw_ls,A,A-n33-k5,32,100,5,446,0.003223,702,661,6.202723,True
3,rfcs,A,A-n33-k5,32,100,5,446,0.056195,668,661,1.059002,True
4,cw_ls,A,A-n33-k6,32,100,6,541,0.007875,743,742,0.134771,True
5,rfcs,A,A-n33-k6,32,100,6,541,0.038068,745,742,0.404313,True
6,cw_ls,A,A-n34-k5,33,100,5,460,0.003781,791,778,1.670951,True
7,rfcs,A,A-n34-k5,33,100,5,460,0.058212,797,778,2.442159,True
8,cw_ls,A,A-n36-k5,35,100,5,442,0.009157,805,799,0.750939,True
9,rfcs,A,A-n36-k5,35,100,5,442,0.059225,831,799,4.005006,True


In [9]:
# Summary: average cost / time / gap / score per solver and size class
results.groupby(['solver'])[['cost', 'time_taken', 'gap_pct']].mean()

,cost,time_taken,gap_pct
solver,,,
cw_ls,1071.148148,0.013465,2.754736
rfcs,1078.777778,0.108949,3.232420


In [13]:
results[['solver', 'feasible']].value_counts()

solver  feasible
cw_ls   True        27
rfcs    True        20
        False        7
Name: count, dtype: int64